# Modeling Experiments.

This notebook summarizes the results generated by `src/models/run_experiments.py`. We will start from the saved artifacts: metrics, summary, predictions, and models to:

1. Visualize and compare metrics by model/fold.
2. Analyze the errors on the test set (residuals, and y_true vs y_pred scatter).
3. Review feature importance (if available for the model) and document conclusions.

### 1. Setup
Update the `EXPERIMENT_DIR` path with the specific folder (e.g., `experiment_YYYYMMDD_HHMMSS`).

In [2]:
import json
from pathlib import Path

import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

EXPERIMENT_DIR = Path("../data/results/modeling/experiments/experiment_20251112_104624")
metrics_path = EXPERIMENT_DIR / "metrics.csv"
summary_path = EXPERIMENT_DIR / "summary.csv"
predictions_path = EXPERIMENT_DIR / "predictions.parquet"
config_path = EXPERIMENT_DIR / "config.json"
feature_cols_path = EXPERIMENT_DIR / "feature_columns.json"
models_dir = EXPERIMENT_DIR  

metrics_df = pd.read_csv(metrics_path)
summary_df = pd.read_csv(summary_path)
pred_df = pd.read_parquet(predictions_path)
if feature_cols_path.exists():
    feature_columns = json.loads(feature_cols_path.read_text())
else:
    feature_columns = None

summary_df


,model,split,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std,samples_total
0,elasticnet,cv,1.205025,0.405229,1.437142,0.404123,0.076414,0.341771,12595
1,elasticnet,test,1.174218,NaN,1.514518,NaN,0.223942,NaN,2954
2,gradient_boosting,cv,1.111449,0.395970,1.322559,0.456710,0.262149,0.101044,12595
3,gradient_boosting,test,1.129386,NaN,1.511933,NaN,0.226589,NaN,2954
4,random_forest,cv,1.125852,0.333313,1.364513,0.384582,0.196147,0.129668,12595
5,random_forest,test,1.202777,NaN,1.668590,NaN,0.058014,NaN,2954


### 2. Metrics Comparison
Charts to compare MAE/RMSE/R² by model and split.

In [3]:
fig = px.bar(
    summary_df,
    x="model",
    y="mae_mean",
    color="split",
    error_y="mae_std",
    title="MAE medio por modelo y split",
)
fig.show()

fig = px.bar(
    summary_df,
    x="model",
    y="rmse_mean",
    color="split",
    error_y="rmse_std",
    title="RMSE medio por modelo y split",
)
fig.show()

fig = px.bar(
    summary_df,
    x="model",
    y="r2_mean",
    color="split",
    error_y="r2_std",
    title="R² medio por modelo y split",
)
fig.show()

### 3. Residuals and scatters (test)

We inspect how each model performs on the test set.

In [4]:
pred_df["residual"] = pred_df["y_true"] - pred_df["y_pred"]

fig = px.scatter(
    pred_df,
    x="y_true",
    y="y_pred",
    color="model",
    title="Dispersión y_true vs y_pred (test)",
    labels={"y_true": "RPE real", "y_pred": "RPE predicho"},
)
fig.add_trace(go.Scatter(x=[pred_df.y_true.min(), pred_df.y_true.max()], y=[pred_df.y_true.min(), pred_df.y_true.max()], mode="lines", name="Ideal"))
fig.show()

fig = px.box(
    pred_df,
    x="model",
    y="residual",
    title="Distribución de residuos por modelo (test)",
)
fig.show()

### 4. Feature Importance




In [5]:
import joblib

gbm_path = models_dir / "gradient_boosting_best.joblib"
if gbm_path.exists():
    gbm = joblib.load(gbm_path)
    importances = gbm.named_steps["model"].feature_importances_
    features = feature_columns if feature_columns is not None else [f"f{i}" for i in range(len(importances))]
    fi = pd.DataFrame({"feature": features, "importance": importances}).sort_values("importance", ascending=False)
    px.bar(fi.head(20), x="feature", y="importance", title="Top features (GradientBoosting)").show()
else:
    print("Archivo gradient_boosting_best.joblib no encontrado.")
